In [1]:
# Instalando as dependencias
%pip install anthropic python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Carregando variaveis .env
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
# Criando o client
from anthropic import Anthropic

client = Anthropic()

In [4]:
# Funcões utilitarias
def add_user_message(messages, text):
    user_message = {
        "role": "user",
        "content": text,
    }
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {
        "role": "assistant",
        "content": text,
    }
    messages.append(assistant_message)

In [5]:
def chat(messages, system_prompt=None):

    params = {
        # Info claude-sonnet-4-0' is deprecated and will reach end-of-life on June 15th, 2026.
        "model": "claude-opus-4-6",
        "max_tokens": 1024,
        "messages": messages,
        # "stop_sequences": stop_sequences
    }

    if system_prompt:
        params["system"] = system_prompt

    # Doc ref: https://platform.claude.com/docs/en/api/python/messages/create
    message = client.messages.create(**params)

    return message.content[0].text

In [6]:
def generate_dataset():
    prompt = """
                Gere um conjunto de dados de avaliação para uma avaliação de prompt. O conjunto de dados será usado para avaliar prompts que geram Python, JSON ou Regex especificamente para tarefas relacionadas à AWS. Gere um array de objetos JSON, cada um representando uma tarefa que requer Python, JSON ou um Regex para ser concluída.

                Exemplo de saída deve ser 
                - um json puro sem formatação markdown
                - sem caracteres de quebra de linha por exemplo '\n'
                - sem aspas duplas ou simples no começo e no fim do json. 
                - Exemplo errado "{"k": "v"}" 
                - Exemplo correto {"k": "v"}
                - Exemplo para json de criaçã de tarefa
                [
                    {
                        "task": "Descrição da tarefa"
                    }
                ]
                
                Foque em tarefas que possam ser resolvidas escrevendo uma única função Python, um único objeto JSON

                Foque em tarefas que não exijam a escrita de muito código

                Por favor, gere 3 objetos.
            """

    messages = []
    add_user_message(messages=messages, text=prompt)
    add_assistant_message(messages=messages, text="")
    
    return chat(messages=messages)

In [7]:
import json

dataset_raw = generate_dataset()

dataset_clean = dataset_raw[1:-1].replace('\\"', '"')

dataset_obj = json.loads(f"[{dataset_clean}]")

with open("dataset.json", "w", encoding='utf-8') as f:
    json.dump(dataset_obj, f, indent=2, ensure_ascii=False)

In [8]:
def grade_by_model(test_case, output):
    eval_prompt = f"""
                        Você é um revisor de código AWS especialista. Sua tarefa é avaliar a seguinte solução gerada por IA.

                        Tarefa Original:
                        <task>
                        {test_case["task"]}
                        </task>

                        Solução a Avaliar:
                        <solution>
                        {output}
                        </solution>

                        Formato de Saída:
                        Forneça sua avaliação como um objeto JSON estruturado com os seguintes campos, nesta ordem específica:
                        - "strengths": Um array com 1 a 3 pontos fortes principais (em português: "pontos_fortes")
                        - "weaknesses": Um array com 1 a 3 áreas principais para melhoria (em português: "pontos_fracos")
                        - "reasoning": Uma explicação concisa da sua avaliação geral (em português: "raciocinio")
                        - "score": Um número entre 1 e 10 (em português: "nota")

                        Responda apenas com JSON. Mantenha sua resposta concisa e direta.
                        Exemplo do formato da resposta:
                        {{
                            "strengths": string[],
                            "weaknesses": string[],
                            "reasoning": string,
                            "score": number
                        }}
                    """

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "")
    
    eval_text = chat(messages)
    
    return eval_text

In [9]:
import json
import re

def extract_score(text):
    try:
        clean_json = re.sub(r'```json|```', '', text).strip()
        
        data = json.loads(clean_json)
        
        return data.get("score")
    
    except Exception as e:
        print(f"Erro ao processar o JSON: {e}")
        return None

In [10]:
from statistics import mean


def run_prompt(test_case):
    """Mescla o prompt e a entrada do caso de teste, então retorna o resultado"""
    prompt = f"""
                Porfavorresolva essa seguinte task

                {test_case["task"]}
                
            """
    messages = []
    add_user_message(messages=messages, text=prompt)
    return chat(messages=messages)


In [11]:
def run_test_case(test_case):
    """Chama run_prompt, então avalia o resultado"""
    output = run_prompt(test_case=test_case)

    model_grade = grade_by_model(test_case=test_case, output=output)

    score = extract_score(model_grade)

    return {"output": output, "test_case": test_case, "score": score}


In [12]:
def run_eval(dataset):
    """Carrega o conjunto de dados e chama run_test_case para cada caso"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case=test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])

    return average_score

In [ ]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset=dataset)


# esta requisição leva mais de 1 minuto
print(results)

7.666666666666667
